# 00 — Джерела та історичний знімок

## Мета

Завантажити всі кандидатні CSV із Kaggle, вибрати файл за **фактичним**, а не заявленим діапазоном дат, дозібрати MSFT і SPY з Nasdaq та зафіксувати незмінний знімок.

**Історичний знімок** — це копія даних на конкретний момент із URL, датою отримання, діапазоном і SHA-256. Завдяки цьому можна довести, на яких саме даних навчалася модель.

Джерела:

- Kaggle: https://www.kaggle.com/datasets/matiflatif/microsoft-complete-stocks-dataweekly-updated
- Nasdaq MSFT: https://www.nasdaq.com/market-activity/stocks/msft/historical
- Nasdaq SPY: https://www.nasdaq.com/market-activity/etf/spy/historical


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Magnitol110/MarketLens.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_path = Path("/content/MarketLens")
    if not repo_path.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)
    os.chdir(repo_path)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "README.md").exists(), f"Repository root not found: {ROOT}"
print("Repository:", ROOT)
print("Running in Colab:", IN_COLAB)

Repository: /content/MarketLens
Running in Colab: True


In [3]:
from datetime import date, datetime, timedelta, timezone
import hashlib
import json
import shutil

import pandas as pd

RAW = ROOT / "data" / "raw"
DOCS = ROOT / "docs"
RAW.mkdir(parents=True, exist_ok=True)
REFRESH_FROM_WEB = IN_COLAB
print("Refresh from web:", REFRESH_FROM_WEB)

Refresh from web: True


## 1. Завантаження та порівняння Kaggle CSV

У Colab notebook автоматично встановлює `kagglehub` і завантажує публічний набір. Локально він використовує вже зафіксований raw-файл.


In [4]:
if REFRESH_FROM_WEB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub
    kaggle_dir = Path(kagglehub.dataset_download("matiflatif/microsoft-complete-stocks-dataweekly-updated"))
    candidate_files = sorted(kaggle_dir.glob("*.csv"))
else:
    candidate_files = sorted(RAW.glob("msft_kaggle_*.csv"))

assert candidate_files, "Kaggle CSV files were not found"

profiles = []
for path in candidate_files:
    frame = pd.read_csv(path)
    date_column = next(column for column in frame.columns if column.strip().lower() == "date")
    parsed_dates = pd.to_datetime(frame[date_column].astype(str).str[:10], errors="coerce")
    profiles.append({
        "path": path,
        "file": path.name,
        "rows": len(frame),
        "valid_rows": int(parsed_dates.notna().sum()),
        "bad_dates": int(parsed_dates.isna().sum()),
        "min_date": parsed_dates.min(),
        "max_date": parsed_dates.max(),
    })

profile = pd.DataFrame(profiles).sort_values(
    ["min_date", "max_date", "valid_rows"], ascending=[True, False, False]
).reset_index(drop=True)
display(profile.drop(columns="path"))

selected = profile.iloc[0]
selected_raw = RAW / "msft_kaggle_1986-03-13_2025-07-11.csv"
if Path(selected["path"]).resolve() != selected_raw.resolve():
    shutil.copy2(selected["path"], selected_raw)
print("Selected:", selected["file"], selected["min_date"].date(), selected["max_date"].date())

100%|██████████| 1.56M/1.56M [00:00<00:00, 118MB/s]

Extracting files...


,file,rows,valid_rows,bad_dates,min_date,max_date
0,MSFT_1986-03-13_2025-07-14.csv,9910,9909,1,1986-03-13,2025-07-11
1,MSFT_1986-03-13_2025-03-02.csv,9818,9818,0,1986-03-13,2025-02-28
2,MSFT_1986-03-13_2025-02-24.csv,9813,9813,0,1986-03-13,2025-02-21
3,MSFT_1986-03-13_2025-02-07.csv,9803,9803,0,1986-03-13,2025-02-06
4,MSFT_1986-03-13_2025-01-31.csv,9798,9798,0,1986-03-13,2025-01-30
5,MSFT_1973-05-08_2025-03-15.csv,7299,7299,0,1996-03-13,2025-03-14
6,MSFT_1996-03-13_2025-02-18.csv,7280,7280,0,1996-03-13,2025-02-14


Selected: MSFT_1986-03-13_2025-07-14.csv 1986-03-13 2025-07-11


## 2. Актуальне доповнення Nasdaq

API-запит використовує ISO-дати. Якщо Nasdaq тимчасово недоступний, повторіть цю комірку пізніше; старий raw-знімок не перезаписується помилковою відповіддю.


In [5]:
def download_nasdaq(symbol, asset_class, start_date, end_date, output_path):
    url = f"https://api.nasdaq.com/api/quote/{symbol}/historical"
    from urllib.parse import urlencode
    from urllib.request import Request, urlopen

    query = urlencode({
        "assetclass": asset_class,
        "fromdate": start_date.isoformat(),
        "todate": end_date.isoformat(),
        "limit": 10000,
    })
    request = Request(
        f"{url}?{query}",
        headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json, text/plain, */*"},
    )
    with urlopen(request, timeout=60) as response:
        payload = json.load(response)
    rows = ((payload.get("data") or {}).get("tradesTable") or {}).get("rows")
    if not rows:
        raise RuntimeError(f"Nasdaq returned no rows for {symbol}: {payload.get('status')}")
    output_path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    return len(rows)

today = date.today()
existing_msft = sorted(RAW.glob("msft_nasdaq_*.json"))
existing_spy = sorted(RAW.glob("spy_nasdaq_*.json"))
msft_json = RAW / f"msft_nasdaq_2025-07-12_{today.isoformat()}.json" if REFRESH_FROM_WEB else existing_msft[-1]
spy_json = RAW / f"spy_nasdaq_recent_{today.isoformat()}.json" if REFRESH_FROM_WEB else existing_spy[-1]

if REFRESH_FROM_WEB:
    msft_rows = download_nasdaq("MSFT", "stocks", date(2025, 7, 12), today, msft_json)
    spy_rows = download_nasdaq("SPY", "etf", today - timedelta(days=3653), today, spy_json)
    print("Downloaded Nasdaq rows:", {"MSFT": msft_rows, "SPY": spy_rows})
else:
    assert existing_msft and existing_spy, "Existing Nasdaq raw snapshots are missing"
    print("Using existing Nasdaq snapshots")

Downloaded Nasdaq rows: {'MSFT': 252, 'SPY': 2513}


## 3. Маніфест знімка

SHA-256 зміниться, якщо хоча б один байт сирого файлу буде змінено.


In [6]:
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

snapshot_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "sources": [
        {
            "name": "Kaggle Microsoft Complete Stocks Data",
            "url": "https://www.kaggle.com/datasets/matiflatif/microsoft-complete-stocks-dataweekly-updated",
            "raw_file": str(selected_raw.relative_to(ROOT)).replace("\\", "/"),
            "sha256": file_sha256(selected_raw),
            "actual_min_date": str(selected["min_date"].date()),
            "actual_max_date": str(selected["max_date"].date()),
        },
        {
            "name": "Nasdaq MSFT Historical",
            "url": "https://www.nasdaq.com/market-activity/stocks/msft/historical",
            "raw_file": str(msft_json.relative_to(ROOT)).replace("\\", "/"),
            "sha256": file_sha256(msft_json),
        },
        {
            "name": "Nasdaq SPY Historical",
            "url": "https://www.nasdaq.com/market-activity/etf/spy/historical",
            "raw_file": str(spy_json.relative_to(ROOT)).replace("\\", "/"),
            "sha256": file_sha256(spy_json),
        },
    ],
}
(DOCS / "data-snapshot.json").write_text(
    json.dumps(snapshot_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(pd.DataFrame(snapshot_manifest["sources"]))

,name,url,raw_file,sha256,actual_min_date,actual_max_date
0,Kaggle Microsoft Complete Stocks Data,https://www.kaggle.com/datasets/matiflatif/mic...,data/raw/msft_kaggle_1986-03-13_2025-07-11.csv,352ad480d66690ccd42bb65b5c63110aaa431fa837ac4b...,1986-03-13,2025-07-11
1,Nasdaq MSFT Historical,https://www.nasdaq.com/market-activity/stocks/...,data/raw/msft_nasdaq_2025-07-12_2026-07-15.json,681e995d2cb9582d74a5c39e2a20048173aff660aa41ef...,NaN,NaN
2,Nasdaq SPY Historical,https://www.nasdaq.com/market-activity/etf/spy...,data/raw/spy_nasdaq_recent_2026-07-15.json,3a58eb8b4bf270817632420ccf6e73a0542c04f6a6f0c1...,NaN,NaN


## 4. Побудова чистих таблиць

Ця команда видаляє службовий рядок, нормалізує OHLCV, об'єднує історію та перевіряє ціни, обсяг і дублікати дат.


In [7]:
sys.path.insert(0, str(ROOT))
from scripts.build_market_dataset import main as build_market_dataset

build_market_dataset()

{
  "msft": {
    "rows": 10161,
    "min_date": "1986-03-13",
    "max_date": "2026-07-14"
  },
  "spy": {
    "rows": 2512,
    "min_date": "2016-07-14",
    "max_date": "2026-07-14"
  },
  "model_table": {
    "rows": 2512,
    "min_date": "2016-07-14",
    "max_date": "2026-07-14"
  }
}


In [8]:
for filename in ["msft_daily.csv", "spy_daily.csv", "msft_spy_daily.csv"]:
    frame = pd.read_csv(ROOT / "data" / "processed" / filename)
    print(filename, {
        "rows": len(frame),
        "first_date": frame["date"].min(),
        "last_date": frame["date"].max(),
        "missing": int(frame.isna().sum().sum()),
        "duplicate_dates": int(frame["date"].duplicated().sum()),
    })


msft_daily.csv {'rows': 10161, 'first_date': '1986-03-13', 'last_date': '2026-07-14', 'missing': 0, 'duplicate_dates': 0}
spy_daily.csv {'rows': 2512, 'first_date': '2016-07-14', 'last_date': '2026-07-14', 'missing': 0, 'duplicate_dates': 0}
msft_spy_daily.csv {'rows': 2512, 'first_date': '2016-07-14', 'last_date': '2026-07-14', 'missing': 0, 'duplicate_dates': 0}


## Результат

Після успішного виконання збережіть оновлений notebook назад у GitHub. Сирі файли залишаються локальними, а `docs/data-snapshot.json` і чисті CSV фіксують доказ використаного знімка.
